# 15쪽 · 메모리 세 경우 비교

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. 준비와 도구 함수

함수는 여기에서 직접 확인할 수 있습니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition


def add_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 더합니다."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """두 정수 a와 b를 곱합니다."""
    return a * b

## 2. 모델에 도구 연결

한 번의 응답에서 도구를 순차적으로 요청하도록 설정합니다.

In [ ]:
llm = make_llm()

tools = [add_numbers, multiply_numbers]
tool_llm = llm.bind_tools(
    tools,
    parallel_tool_calls=False,
)

## 3. Assistant 정의

도구 결과를 받은 뒤에도 같은 tool_llm을 사용합니다.

In [ ]:
system = SystemMessage(
    content=(
        "계산은 제공된 도구를 사용하세요. "
        "도구 결과를 확인하고 필요한 계산을 이어서 수행하세요. "
        "계산이 끝나면 한국어로 답하세요. "
        "이전 결과를 모르면 추측하지 말고 물어보세요."
    )
)


def assistant(state: MessagesState):
    messages = [system] + state["messages"]
    response = tool_llm.invoke(messages)

    return {"messages": [response]}

## 4. 그래프 연결

tools에서 assistant로 돌아오는 한 줄이 핵심입니다.

In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)

# 도구 결과를 모델에게 돌려보냅니다.
builder.add_edge("tools", "assistant")

## A-1. 메모리 없음

A는 저장하지 않는 그래프입니다.

In [ ]:
graph_a = builder.compile()

### A 그래프 보기

In [ ]:
from lab.display import show_mermaid

show_mermaid(graph_a)

## A-2. 첫 질문

3과 4를 더합니다.

In [ ]:
messages = [
    HumanMessage(content="3과 4를 더해줘."),
]

first_a = graph_a.invoke({"messages": messages})

for message in first_a["messages"]:
    message.pretty_print()

## A-3. 두 번째 질문

이전 결과를 가리켜봅니다.

In [ ]:
messages = [
    HumanMessage(content="그 결과에 2를 곱해줘."),
]

second_a = graph_a.invoke({"messages": messages})

for message in second_a["messages"]:
    message.pretty_print()

## B-1. 메모리 있음, 같은 thread

새 메모리를 만들고 대화 번호를 1로 정합니다.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory_b = MemorySaver()
graph_b = builder.compile(checkpointer=memory_b)

config_b = {
    "configurable": {
        "thread_id": "1",
    },
}

### B 그래프 보기

In [ ]:
from lab.display import show_mermaid

show_mermaid(graph_b)

## B-2. 첫 질문

thread 1을 사용합니다.

In [ ]:
messages = [
    HumanMessage(content="3과 4를 더해줘."),
]

first_b = graph_b.invoke(
    {"messages": messages},
    config_b,
)

for message in first_b["messages"]:
    message.pretty_print()

## B-3. 두 번째 질문

그대로 thread 1을 사용합니다.

In [ ]:
messages = [
    HumanMessage(content="그 결과에 2를 곱해줘."),
]

second_b = graph_b.invoke(
    {"messages": messages},
    config_b,
)

for message in second_b["messages"]:
    message.pretty_print()

## C-1. 메모리 있음, 다른 thread

B와 구분하기 위해 메모리도 새로 만듭니다.

In [ ]:
memory_c = MemorySaver()
graph_c = builder.compile(checkpointer=memory_c)

config_1 = {
    "configurable": {
        "thread_id": "1",
    },
}

config_2 = {
    "configurable": {
        "thread_id": "2",
    },
}

### C 그래프 보기

In [ ]:
from lab.display import show_mermaid

show_mermaid(graph_c)

## C-2. 첫 질문은 thread 1

첫 번째 질문에 config_1을 사용합니다.

In [ ]:
messages = [
    HumanMessage(content="3과 4를 더해줘."),
]

first_c = graph_c.invoke(
    {"messages": messages},
    config_1,
)

for message in first_c["messages"]:
    message.pretty_print()

## C-3. 두 번째 질문은 thread 2

다른 대화 번호를 사용합니다.

In [ ]:
messages = [
    HumanMessage(content="그 결과에 2를 곱해줘."),
]

second_c = graph_c.invoke(
    {"messages": messages},
    config_2,
)

for message in second_c["messages"]:
    message.pretty_print()

## A 결과의 사용자 메시지만 확인

다른 경우와 섞이지 않도록 각각 출력합니다.

In [ ]:
for message in second_a["messages"]:
    if isinstance(message, HumanMessage):
        print(message.content)

## B 결과의 사용자 메시지만 확인

다른 경우와 섞이지 않도록 각각 출력합니다.

In [ ]:
for message in second_b["messages"]:
    if isinstance(message, HumanMessage):
        print(message.content)

## C 결과의 사용자 메시지만 확인

다른 경우와 섞이지 않도록 각각 출력합니다.

In [ ]:
for message in second_c["messages"]:
    if isinstance(message, HumanMessage):
        print(message.content)